# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vinay21rout/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content item per day in the daily performance table. I will use February 2026 as the feature window and March 2026 as the future label window.

In [12]:
import os
import getpass
import duckdb

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{get_hf_token()}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to FlyRank warehouse")
print("Feature window: February 2026")
print("Label window: March 2026")

Paste your Hugging Face READ token (hf_...): ··········
Connected to FlyRank warehouse
Feature window: February 2026
Label window: March 2026


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: February impressions, February clicks, February average position, February sessions, and February content age.

Label: March CTR / performance outcome, used to identify pages with weaker future search performance.

Context: client identifier, content identifier, report dates, and data-availability fields used to join, group, filter, and validate the data.

Excluded: March clicks, March impressions, March sessions, and other future outcome fields are excluded from the features because they would not be available at the February prediction cutoff. IDs are used only for joins/grouping, not as predictive features.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
# Check the grain: one row per client + content + report date

grain_check = con.execute(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS row_count
    FROM {FEB}
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchdf()

grain_check

,client_hash_id,content_hash_id,report_date,row_count


In [16]:
# Check row counts and date windows for the feature and label periods

window_check = con.execute(f"""
    SELECT
        'February 2026' AS window,
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {FEB}

    UNION ALL

    SELECT
        'March 2026' AS window,
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {MAR}
""").fetchdf()

window_check

,window,row_count,min_date,max_date
0,February 2026,7355108,2026-02-01,2026-02-28
1,March 2026,9841378,2026-03-01,2026-03-31


In [17]:
# Check the actual column names in the daily performance table

columns_check = con.execute(f"""
    DESCRIBE SELECT *
    FROM {FEB}
""").fetchdf()

columns_check

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [18]:
# Check missing values across the February feature window

schema = con.execute(f"""
    DESCRIBE SELECT *
    FROM {FEB}
""").fetchdf()

missing_parts = []

for col in schema["column_name"]:
    safe_col = '"' + col.replace('"', '""') + '"'
    missing_parts.append(
        f"AVG(CASE WHEN {safe_col} IS NULL THEN 1.0 ELSE 0 END) AS \"{col}\""
    )

missing_sql = f"""
    SELECT
        {", ".join(missing_parts)}
    FROM {FEB}
"""

missing_check = con.execute(missing_sql).fetchdf()
missing_check.T

,0
report_date,0.000000
client_hash_id,0.000000
content_hash_id,0.000000
client_has_gsc,0.000000
client_has_ga4,0.000000
gsc_data_available,0.012543
ga4_data_available,0.564550
gsc_impressions,0.012543
gsc_clicks,0.012543
gsc_sum_position,0.012543


In [19]:
# Check whether GSC missingness follows the data-availability flag

gsc_missing_pattern = con.execute(f"""
    SELECT
        gsc_data_available,
        COUNT(*) AS rows,
        AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS impressions_missing,
        AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS clicks_missing,
        AVG(CASE WHEN gsc_sum_position IS NULL THEN 1.0 ELSE 0 END) AS position_missing
    FROM {FEB}
    GROUP BY gsc_data_available
    ORDER BY gsc_data_available
""").fetchdf()

gsc_missing_pattern

,gsc_data_available,rows,impressions_missing,clicks_missing,position_missing
0,False,4641069,0.0,0.0,0.000000e+00
1,True,2621783,0.0,0.0,3.814198e-07
2,<NA>,92256,1.0,1.0,1.000000e+00


In [20]:
# Verify the feature window and future label window

label_window_check = con.execute(f"""
    SELECT
        'February feature window' AS period,
        COUNT(*) AS rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {FEB}

    UNION ALL

    SELECT
        'March label window' AS period,
        COUNT(*) AS rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {MAR}
""").fetchdf()

label_window_check

,period,rows,min_date,max_date
0,February feature window,7355108,2026-02-01,2026-02-28
1,March label window,9841378,2026-03-01,2026-03-31


Data limits: The February 2026 feature window is used to define information available before prediction. The March 2026 window is used only for the future outcome/label. Search data can be missing when GSC data is unavailable, and GA4 availability varies across rows. IDs are used for joining and grouping, not as model features.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [23]:
data_limits_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available = FALSE THEN 1 ELSE 0 END) AS gsc_unavailable_rows,
        SUM(CASE WHEN ga4_data_available = FALSE THEN 1 ELSE 0 END) AS ga4_unavailable_rows
    FROM {FEB}
""").fetchdf()

data_limits_check

,total_rows,gsc_unavailable_rows,ga4_unavailable_rows
0,7355108,4641069.0,3057464.0


Final data contract: One row represents one client-content-day. February 2026 is the feature window, and March 2026 is the future label window. Features must use only information available in February; March outcome fields are excluded from features. GSC and GA4 availability flags must be checked before interpreting missing or zero values.

## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x ] No client names, URLs, or private queries anywhere
- [x ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.